# 1 · Introduction to Pandas for Petrologists

**Module 1 · Data Processing with `petropandas`, part 1**

This notebook is a from-scratch introduction to [pandas](https://pandas.pydata.org/),
the Python library almost everything else today is built on. No prior Python
or pandas experience is assumed.

We will use two real datasets that ship *inside* the `petropandas` package
(installed during this morning's setup session), so every cell below runs
immediately — no extra files to download.

**By the end of this notebook you will be able to:**
- load a CSV file into a pandas `DataFrame`
- inspect, filter and select rows/columns
- add new calculated columns
- summarise data with `.groupby()`
- save your results back to a CSV file

Notebook 2 (`02_petropandas.ipynb`) picks up exactly where this one ends and
introduces `petropandas`'s own petrology-aware extensions (mineral formula
recalculation, end-members, bulk-rock tools). Everything you learn here about
plain pandas still applies there — `petropandas` DataFrames *are* pandas
DataFrames.


## Before you start

Jupyter notebooks alternate two kinds of cells:

- **markdown cells** — headings and explanations, like this one. Double-click
  one to edit it, then `Shift+Enter` to render it.
- **code cells** — Python you actually run. Click a code cell and press
  `Shift+Enter` to execute it; the output appears right below.

Two rules for today:

- **Run cells strictly top-to-bottom, in order.** Cells in a notebook share
  memory: a later cell often uses variables created by an earlier one. Running
  cells out of order is the #1 reason a demo works for the lecturer but not for
  you.
- **If something looks broken later**, do `Kernel → Restart & Run All…` — it
  wipes the kernel's memory and re-runs every cell cleanly, top to bottom.

A note on the packages: this notebook uses the **standard** `import pandas as pd`.
From Notebook 2 on we'll use `from petropandas import pd`, a line that re-exports
pandas itself *and* adds petropandas' petrology-aware accessors — everything you
learn here about plain pandas transfers unchanged, because a `petropandas`
DataFrame is still a completely ordinary pandas DataFrame.


In [ ]:
import pandas as pd

# Common display options (optional)
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

print("pandas version:", pd.__version__)

## 1.1 DataFrames and Series

A pandas **`DataFrame`** is a table: rows are observations (here: individual
EPMA analyses), columns are variables (oxide weight percentages, sample
metadata, ...). A single column pulled out of a DataFrame is a **`Series`**.

We use one dataset bundled with `petropandas`:

- `minerals` — 315 real electron-microprobe mineral analyses spanning 21
  mineral groups (garnet, amphibole, pyroxene, mica, ...) from a range of
  metamorphic rocks.

In [ ]:
from petropandas.data import minerals
minerals

A single column is a `Series` — e.g. all the SiO2 values as one 1-D array
with an index:


In [ ]:
sio2 = minerals["SiO2"]
sio2


## 1.2 Data inspection and selection

Before doing anything else with a new dataset, always look at it:

- `.shape` — (rows, columns)
- `.head()` / `.tail()` — first / last rows
- `.describe()` — quick statistics (mean, std, min/max, quartiles) for every
  numeric column
- `.dtypes` — column data types


In [ ]:
print("shape:", minerals.shape)
print("columns:", list(minerals.columns))


In [ ]:
minerals.describe()

`minerals["Mineral"]` tells us which of the 21 mineral each row belongs to. To count number of occurences, we can use `value_counts` method.


In [ ]:
minerals["Mineral"].value_counts()

Selecting rows by a condition ("boolean indexing") is the single
most useful pandas operation for a petrologist: *give me only the garnet
analyses*.

In [ ]:
is_garnet = minerals["Mineral"] == "Garnet"
garnet = minerals[is_garnet]
garnet.head()


Conditions combine with `&` (and) / `|` (or) — each condition needs its own
parentheses. Here: garnet analyses from granulite-facies rocks only.


In [ ]:
is_garnet_granulite = (minerals["Mineral"] == "Garnet") & (minerals["Metamorphic_Grade"] == "Granulite Facies")
garnet_granulite = minerals[is_garnet_granulite]
garnet_granulite.head()


## 1.3 Data manipulation: calculated columns

Adding a new column is just an assignment — pandas applies the arithmetic
row-by-row automatically ("vectorised" operations, no manual loop needed).

A classic petrological ratio is **Fe/(Fe+Mg)** ("Fe-number"). Here we compute
a simplified version straight from oxide weight percentages. This is *not*
the correct molar Fe/(Fe+Mg): FeO and MgO have different molar masses, so a
wt%-based ratio is only a rough proxy. Getting a proper molar/cation-based
ratio needs `petropandas`'s `.moles`/`.cations` accessors — that's exactly
where Notebook 2 picks up.


In [ ]:
garnet = garnet.copy()  # work on an independent copy, leaving the filtered
                        # `minerals` subset above untouched
garnet["Fe_ratio"] = garnet["FeO"] / (garnet["FeO"] + garnet["MgO"])
garnet[["Analysis_ID", "FeO", "MgO", "Fe_ratio"]].head()

## 1.4 Aggregation and grouping

`.groupby()` splits a DataFrame into groups sharing a value, applies a
function to each group, and combines the results back into one table. This
is how you would compute e.g. the average composition per mineral group
across the whole 315-analysis dataset.

(Our `minerals` dataset has one analysis per row across many different rock
groups rather than a repeated per-sample structure, so "average composition
per mineral group" is the meaningful grouping here — Notebook 2's garnet
zoning profile is where you will group/average *within one sample*.)


In [ ]:
mean_by_mineral = minerals.groupby("Mineral").mean(numeric_only=True)
with pd.option_context('display.max_rows', None):
    display(mean_by_mineral)


## 1.5 Exporting data

Once you have filtered/derived the subset you need, `.to_csv()` writes it
back out — ready to hand to a colleague or feed into another tool.


In [ ]:
out_cols = ["Analysis_ID", "Rock_Type", "Metamorphic_Grade",
            "SiO2", "TiO2", "Al2O3", "Cr2O3", "Fe2O3", "FeO", "MnO",
            "MgO", "CaO", "Na2O", "K2O", "Fe_ratio"]

garnet[out_cols].to_csv("garnet_analyses_filtered.csv", index=False)
print("Saved garnet_analyses_filtered.csv")

## 1.6 Hands-on exercise

Using the `minerals` DataFrame:

1. Select all **Garnet** analyses into a new DataFrame `df_grt`.
2. Add a column `Fe_ratio` = `FeO / (FeO + MgO)`.
3. Compute the **average** `Fe_ratio` grouped by `Rock_Type`.
4. Find the single garnet analysis with the **highest MgO** (hint:
   `.sort_values()` or `.idxmax()`).

Try it yourself in the cell below before looking at the solution.


In [ ]:
# Your code here
df_grt = ...



<details><summary><b>Solution (click to expand)</b></summary>

Try it yourself first — then compare with this:

```python
df_grt = minerals[minerals["Mineral"] == "Garnet"].copy()
df_grt["Fe_ratio"] = df_grt["FeO"] / (df_grt["FeO"] + df_grt["MgO"])

print(df_grt.groupby("Rock_Type")["Fe_ratio"].mean())

highest_mg_row = df_grt.loc[df_grt["MgO"].idxmax()]
print("\nHighest-MgO garnet analysis:")
print(highest_mg_row[["Analysis_ID", "Rock_Type", "MgO"]])
```

Expected output — the highest-MgO garnet analysis is `GA-ECL-01`
(12.09 wt% MgO, Mafic Eclogite).

</details>


## Recap

- A pandas `DataFrame` is a table; a `Series` is one column of it.
- `df[condition]` filters rows; `df["col"]` / `df[["c1", "c2"]]` select
  columns.
- New columns are plain arithmetic assignments — pandas vectorises them.
- `.groupby("col").mean()` (or `.sum()`, `.describe()`, ...) summarises by
  category.
- `.to_csv()` writes a DataFrame back to disk.

**Next up:** `02_petropandas.ipynb` — recalculating mineral structural
formulas, end-members, and bulk-rock compositions with `petropandas`'s
domain-specific pandas accessors.


## Common pitfalls

- **Compound conditions need parentheses around each test**, and `&` (and) / `|`
  (or) instead of the English words:
  `(minerals["Mineral"] == "Garnet") & (minerals["Metamorphic_Grade"] == "Amphibolite Facies")`.
- **A typo in a column name raises `KeyError`.** If you hit one, check
  `df.columns` for the exact spelling.
- **Brackets select rows *or* columns, and the two look very similar:**
  `df["col"]` pulls out one **column**; `df[boolean_condition]` filters
  **rows** (e.g. the `True`/`False` Series from section 1.2).
- **Memory is shared across cells.** If a variable suddenly seems "gone", you
  almost certainly ran cells out of order — do `Kernel → Restart & Run All`.
